# Notebook B

Run every cell **top to bottom in a fresh kernel**. Setup follows Notebook A: skips Git LFS downloads, keeps Kaggle's PyTorch installation, discovers the attached CSV/checkpoint, and supports graphs without native PyG extensions.

Train the edge policy with a frozen GNN. Optional Stage B fine-tuning uses a separate model, while policy embeddings always come from the original frozen model. Download this notebook's output archive and attach its contents to C and D.

Set `SMOKE_TEST=True` for a small pipeline check; smoke outputs live in a separate folder and cannot be mixed with full results. Use the same dataset, checkpoint and split settings in all notebooks. Attach complete output folders (including `provenance.json`) between stages. Set explicit artifact paths if more than one run is attached. Older B outputs must be regenerated.

Default repository branch: `fix/rl-pruning-symmetry`. Restart the kernel after syncing changed Python modules. Optional ablations and CAMELS run only when enabled in settings.

The repaired policy uses sampled physical pairs (Plackett–Luce), train-fitted input normalization and reward scaling. Best weights are selected on validation at the final pair budget. Failed policies are saved as policy_diagnostic.pt and cannot be loaded for full downstream runs. Stage B saves finetuned_gnn.pt only for an accepted validation improvement.


In [1]:
# Settings — edit these before Run All.
import os
import sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/cosmic-net")
REPO_URL = "https://github.com/Neal-Salian/cosmic-net-f.git"
REPO_BRANCH = "fix/rl-pruning-symmetry"
SYNC_REPO = True                 # False for an already prepared local checkout.
INSTALL_MISSING = True
INPUT_DIR = None                 # None: discover one matching Kaggle dataset.
INPUT_ROOT = Path("/kaggle/input")
DEVICE = "auto"                  # "auto", "cpu", or "cuda"
SMOKE_TEST = False
SMOKE_GRAPHS_PER_SPLIT = 3
GRAPH_BACKEND = "auto"           # "auto" or "torch" (no native extensions)
OUTPUT_DIR = None  # Default: outputs/rls/notebook_B
POLICY_EPOCHS = None  # None uses config.yaml; smoke runs cap epochs at two.
POLICY_OVERRIDES = {}  # Training/reward overrides, recorded for C and D.
RUN_STAGE_B = True
STAGEB_EPOCHS = None
RUN_PENALTY_ABLATION = False


In [2]:
# Dependencies — retain Kaggle's PyTorch and install only missing core packages.
import importlib
import importlib.util
import subprocess
import torch

print("Python:", sys.version.split()[0], "| torch:", torch.__version__,
      "| CUDA build:", torch.version.cuda, "| GPU available:", torch.cuda.is_available())
packages = {"torch_geometric": "torch-geometric>=2.6,<3", "yaml": "pyyaml",
            "numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
            "matplotlib": "matplotlib", "h5py": "h5py", "requests": "requests", "dotenv": "python-dotenv"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    if not INSTALL_MISSING:
        raise RuntimeError("Missing packages: " + ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
    importlib.invalidate_caches()
# NNConv works without torch-scatter/torch-sparse. Do not install CPU extension
# wheels into a CUDA runtime or compile unsupported wheels during notebook setup.
import torch_geometric
print("PyG:", torch_geometric.__version__)


Python: 3.12.13 | torch: 2.10.0+cu128 | CUDA build: 12.8 | GPU available: True
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.3 MB/s eta 0:00:00
PyG: 2.8.0.post1


In [3]:
# Repository checkout — skip LFS and never put the PAT in command-line URLs.
import base64
import tempfile

def prepare_repository(repo_dir, repo_url, branch, sync=True):
    repo_dir = Path(repo_dir).resolve()
    git_env = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1", GIT_TERMINAL_PROMPT="0")
    token = ""
    encoded = ""
    if sync:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("GITHUB_PAT") or ""
        except Exception:
            # Public repositories can be cloned without a secret.
            print("GITHUB_PAT unavailable; trying unauthenticated repository access.")
        if token:
            encoded = base64.b64encode(("x-access-token:" + token).encode()).decode()
            # Git reads this transient header from the child environment only.
            count = int(git_env.get("GIT_CONFIG_COUNT", "0"))
            git_env.update({"GIT_CONFIG_COUNT": str(count + 1),
                            f"GIT_CONFIG_KEY_{count}": "http.https://github.com/.extraheader",
                            f"GIT_CONFIG_VALUE_{count}": "Authorization: Basic " + encoded})

    def git(*args):
        result = subprocess.run(["git", *map(str, args)], env=git_env,
                                capture_output=True, text=True)
        if result.returncode:
            detail = result.stderr or result.stdout
            for secret in (token, encoded):
                if secret:
                    detail = detail.replace(secret, "[redacted]")
            raise RuntimeError("Git operation failed:\n" + detail.strip()) from None
        return result.stdout.strip()

    if sync:
        if not repo_dir.exists():
            repo_dir.parent.mkdir(parents=True, exist_ok=True)
            staging = Path(tempfile.mkdtemp(prefix="cosmic-net-clone-", dir=repo_dir.parent)) / "repo"
            git("clone", "--no-checkout", "--branch", branch, repo_url, staging)
            git("-C", staging, "checkout", branch)
            staging.rename(repo_dir)  # A failed clone never becomes REPO_DIR.
        if not (repo_dir / ".git").exists():
            raise RuntimeError(f"{repo_dir} is not a Git checkout. Select a different REPO_DIR.")
        git("-C", repo_dir, "remote", "set-url", "origin", repo_url)
        git("-C", repo_dir, "fetch", "origin", branch)
        head = git("-C", repo_dir, "rev-parse", "HEAD")
        fetched = git("-C", repo_dir, "rev-parse", "FETCH_HEAD")
        current = git("-C", repo_dir, "branch", "--show-current")
        if head != fetched or current != branch:
            if git("-C", repo_dir, "status", "--porcelain", "--untracked-files=no"):
                raise RuntimeError("Cached checkout has local edits or an incomplete checkout. "
                                   "Preserve it and select a new REPO_DIR before rerunning setup.")
            git("-C", repo_dir, "checkout", branch)
            git("-C", repo_dir, "merge", "--ff-only", "FETCH_HEAD")
    if not (repo_dir / "config/config.yaml").is_file():
        raise FileNotFoundError(f"Repository files missing in {repo_dir}; run checkout first.")
    head = git("-C", repo_dir, "rev-parse", "HEAD")
    if globals().get("_B_IMPORTED_HEAD") not in (None, head):
        raise RuntimeError("Repository revision changed after imports. Restart the kernel and Run All.")
    return repo_dir, head

REPO_DIR, REPO_HEAD = prepare_repository(REPO_DIR, REPO_URL, REPO_BRANCH, SYNC_REPO)
repo_path = str(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
os.chdir(REPO_DIR)
print("Repository:", REPO_DIR, "| HEAD:", REPO_HEAD)


Repository: /kaggle/working/cosmic-net | HEAD: f538726bbc36b99c2c7fd75cd82a48fe3d8b968c


In [4]:
# Locate and validate the attached dataset before any model or graph work.
import numpy as np
import pandas as pd

def locate_inputs(input_dir, input_root):
    if input_dir is not None:
        candidates = [Path(input_dir)]
    else:
        candidates = sorted({p.parent for p in Path(input_root).rglob("tng100_clustered.csv")
                             if (p.parent / "best_model_augmented.pt").is_file()})
    if len(candidates) != 1:
        raise FileNotFoundError("Set INPUT_DIR to the dataset folder containing "
                                "tng100_clustered.csv and best_model_augmented.pt. "
                                f"Matching folders: {candidates}")
    folder = candidates[0].resolve()
    csv_path = folder / "tng100_clustered.csv"
    checkpoint_path = folder / "best_model_augmented.pt"
    for path in (csv_path, checkpoint_path):
        if not path.is_file() or path.stat().st_size == 0:
            raise FileNotFoundError(f"Missing or empty input: {path}")
        with path.open("rb") as stream:
            if stream.read(128).startswith(b"version https://git-lfs.github.com/spec/v1"):
                raise ValueError(f"{path} is a Git LFS pointer, not the actual data file.")
    return csv_path, checkpoint_path

CSV_PATH, CHECKPOINT_PATH = locate_inputs(INPUT_DIR, INPUT_ROOT)
catalog = pd.read_csv(CSV_PATH)
velocity_column = "vel_dispersion" if "vel_dispersion" in catalog else "velocity_dispersion"
required = ["subhalo_id", "group_id", "stellar_mass", velocity_column,
            "half_mass_radius", "metallicity", "pos_x", "pos_y", "pos_z",
            "vel_x", "vel_y", "vel_z"]
target_column = "halo_mass" if "halo_mass" in catalog else "halo_mass_log"
required.append(target_column)
missing_columns = sorted(set(required) - set(catalog.columns))
if missing_columns:
    raise ValueError(f"CSV missing required columns: {missing_columns}")
numeric = catalog[required].apply(pd.to_numeric, errors="coerce")
if catalog.empty or not np.isfinite(numeric.to_numpy()).all():
    raise ValueError("CSV must contain finite numeric features and targets in every row.")
if (numeric[[velocity_column, "half_mass_radius", "metallicity"]] < 0).any().any():
    raise ValueError("Velocity dispersion, radius and metallicity cannot be negative.")
print("CSV:", CSV_PATH, "| rows:", len(catalog), "| halos:", catalog.group_id.nunique())
print("Checkpoint:", CHECKPOINT_PATH, "| bytes:", CHECKPOINT_PATH.stat().st_size)


CSV: /kaggle/input/datasets/nealsalian/cosmicnet-data/tng100_clustered.csv | rows: 5378 | halos: 541
Checkpoint: /kaggle/input/datasets/nealsalian/cosmicnet-data/best_model_augmented.pt | bytes: 3349165


In [5]:
# Configuration and imports — retain the repository's RL settings.
import copy
import yaml
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Batch, Data
from torch_geometric.loader import DataLoader
from data.loaders.base_loader import get_loader
from graph.graph_builder import GraphBuilder
from model.model import build_model
from model.physics_loss import MetricsComputer
from rls.baselines import (random_mask, degree_mask, distance_mask,
                           mass_ratio_mask, gradient_saliency_mask,
                           attention_topk_mask, GumbelEdgeMask)
from rls.sparsify import hard_mask, repair_connectivity
from rls.provenance import record_backbone, require_backbone_label

_B_IMPORTED_HEAD = REPO_HEAD
cfg = yaml.safe_load((REPO_DIR / "config/config.yaml").read_text())
cfg["data"].update(source="tng", num_workers=0, batch_size=16)
cfg["data"]["tng"]["clustered_file"] = str(CSV_PATH)
if DEVICE not in ("auto", "cpu", "cuda"):
    raise ValueError("DEVICE must be auto, cpu or cuda.")
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA requested but unavailable. Enable a Kaggle GPU or use DEVICE='cpu'.")
device = torch.device("cuda" if DEVICE == "cuda" or
                      (DEVICE == "auto" and torch.cuda.is_available()) else "cpu")
if GRAPH_BACKEND not in ("auto", "torch"):
    raise ValueError("GRAPH_BACKEND must be auto or torch.")
if SMOKE_GRAPHS_PER_SPLIT < 2:
    raise ValueError("Use at least two smoke graphs per split.")
OUT = Path(OUTPUT_DIR) if OUTPUT_DIR is not None else REPO_DIR / "outputs/rls/notebook_B"
if SMOKE_TEST:
    OUT = OUT / "smoke"
OUT.mkdir(parents=True, exist_ok=True)
cfg["data"]["tng"]["cache_dir"] = str(OUT / "tng_cache")
# Mark this run as incomplete until the final cell records all artifacts.
import json
provenance_path = OUT / "provenance.json"
previous = json.loads(provenance_path.read_text()) if provenance_path.exists() else {}
previous["B"] = {"stage": "B", "status": "running", "repo": {"head": REPO_HEAD}}
provenance_path.write_text(json.dumps(previous, indent=2))
seed = int(cfg.get("seed", 42))
torch.manual_seed(seed)
np.random.seed(seed)
print("Device:", device, "| run mode:", "SMOKE" if SMOKE_TEST else "FULL", "| outputs:", OUT)


Device: cuda | run mode: FULL | outputs: /kaggle/working/cosmic-net/outputs/rls/notebook_B


In [6]:
# Load only the supplied checkpoint; use its own architecture and graph settings.
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
checkpoint_config = checkpoint.get("config") if isinstance(checkpoint, dict) else None
if checkpoint_config and "model" in checkpoint_config:
    cfg["model"] = copy.deepcopy(checkpoint_config["model"])
    if "graph" in checkpoint_config:
        cfg["graph"] = copy.deepcopy(checkpoint_config["graph"])
cfg["model"]["mc_samples"] = 30
if cfg["graph"].get("hierarchical", False):
    raise ValueError("Notebook B supports single-level graphs; hierarchical mode is only a scaffold.")
state_dict = checkpoint.get("model_state_dict", checkpoint)
gnn = build_model(cfg)
try:
    gnn.load_state_dict(state_dict, strict=True)
except RuntimeError as exc:
    raise RuntimeError("Checkpoint architecture does not match CosmicNetGNN. "
                       "Supply the matching augmented checkpoint/config.\n" + str(exc)) from None
gnn = gnn.to(device).eval()
gnn.requires_grad_(False)  # Saliency still differentiates input features.
model_device = next(gnn.parameters()).device
backbone_record = record_backbone("frozen", gnn)
del checkpoint, state_dict
print("Loaded:", CHECKPOINT_PATH.name, "| dimensions:", gnn.hidden_dim, gnn.output_dim,
      "| parameters:", sum(p.numel() for p in gnn.parameters()))


Loaded: best_model_augmented.pt | dimensions: 64 64 | parameters: 831809


In [7]:
# Build each halo independently; fail on invalid graphs instead of dropping them.
class NotebookGraphBuilder(GraphBuilder):
    """Native PyG when available; chunked pure-torch radius/kNN otherwise."""
    backend_used = "native"

    def _torch_edges(self, positions, method):
        self.backend_used = "torch"
        n = positions.shape[0]
        if n <= 1:
            return torch.empty((2, 0), dtype=torch.long, device=positions.device)
        parts = []
        for start in range(0, n, 256):
            stop = min(start + 256, n)
            distances = torch.cdist(positions[start:stop], positions,
                                    compute_mode="donot_use_mm_for_euclid_dist")
            distances[torch.arange(stop - start, device=positions.device),
                      torch.arange(start, stop, device=positions.device)] = float("inf")
            if method == "radius":
                target, source = (distances < self.radius_mpc).nonzero(as_tuple=True)
                parts.append(torch.stack([source, target + start]))
            else:
                k = min(self.k_neighbors, n - 1)
                if k < 1:
                    raise ValueError("k_neighbors must be positive.")
                source = distances.topk(k, largest=False, dim=1).indices.reshape(-1)
                target = torch.arange(start, stop, device=positions.device).repeat_interleave(k)
                parts.append(torch.stack([source, target]))
        edges = torch.cat(parts, dim=1)
        if method == "knn":
            edges = torch.cat([edges, edges.flip(0)], dim=1)
        return torch.unique(edges, dim=1)

    def _build_radius_edges(self, positions, num_nodes):
        if GRAPH_BACKEND == "torch":
            return self._torch_edges(positions, "radius")
        try:
            return super()._build_radius_edges(positions, num_nodes)
        except (ImportError, OSError):
            return self._torch_edges(positions, "radius")

    def _build_knn_edges(self, positions, num_nodes):
        if GRAPH_BACKEND == "torch":
            return self._torch_edges(positions, "knn")
        try:
            return super()._build_knn_edges(positions, num_nodes)
        except (ImportError, OSError):
            return self._torch_edges(positions, "knn")

def validate_graph(graph):
    if graph.num_nodes < 1 or graph.edge_index.shape[1] == 0:
        raise ValueError(f"Empty graph or no edges: {graph.cluster_id}")
    graph.validate(raise_on_error=True)
    if graph.x.shape[1] != gnn.node_input_dim or graph.edge_attr.shape != (
            graph.edge_index.shape[1], gnn.edge_input_dim):
        raise ValueError(f"Feature dimensions incompatible with checkpoint: {graph.cluster_id}")
    for name in ("x", "pos", "edge_attr", "y"):
        value = getattr(graph, name, None)
        if value is None or not torch.isfinite(value).all():
            raise ValueError(f"Missing or non-finite {name}: {graph.cluster_id}")
    if graph.pos.shape != (graph.num_nodes, 3):
        raise ValueError(f"Expected physical 3D positions: {graph.cluster_id}")

loader = get_loader(cfg)
halos = loader.load()
split_halos = dict(zip(("train", "val", "test"), loader.split_data(halos)))
full_split_sizes = {name: len(items) for name, items in split_halos.items()}
if any(size < 2 for size in full_split_sizes.values()):
    raise ValueError(f"Each split needs at least two halos; got {full_split_sizes}.")
if SMOKE_TEST:
    split_halos = {name: items[:SMOKE_GRAPHS_PER_SPLIT] for name, items in split_halos.items()}
torch.manual_seed(seed)
np.random.seed(seed)
builder = NotebookGraphBuilder(cfg)
split_graphs = {}
for name, items in split_halos.items():
    graphs = []
    for halo in items:
        graph = builder.build_graph(halo)
        validate_graph(graph)
        graphs.append(graph)
    split_graphs[name] = graphs
train_loader = DataLoader(split_graphs["train"], batch_size=cfg["data"]["batch_size"], shuffle=True)
val_loader = DataLoader(split_graphs["val"], batch_size=cfg["data"]["batch_size"], shuffle=False)
test_loader = DataLoader(split_graphs["test"], batch_size=cfg["data"]["batch_size"], shuffle=False)
print("Full split:", full_split_sizes, "| evaluated split:",
      {name: len(graphs) for name, graphs in split_graphs.items()},
      "| graph backend:", builder.backend_used)


TNG_API_KEY not set. Will try local files first.


Full split: {'train': 378, 'val': 81, 'test': 82} | evaluated split: {'train': 378, 'val': 81, 'test': 82} | graph backend: torch


In [8]:
from rls.notebook_workflow import (
    run_context, file_info, checked_artifact, source_artifacts, load_saved_policy,
    prepared_splits, predict_graph, prediction_pair, policy_masks, evaluate_masks,
    result_rows, merge_baselines, physics_diagnostics, coverage_rows,
    adaptation_trial, validation_gates, complete_stage, local_camels_graphs,
    policy_metrics, random_pair_reference, policy_validation)
from rls.evaluate import save_paper_plots
from rls.policy import build_policy
from rls.policy_gradient import ValueNet, PolicyGradientTrainer
from rls.train_policy import train_policy
from rls.stageb import fine_tune_gnn
context = run_context(cfg, CSV_PATH, CHECKPOINT_PATH, split_graphs, backbone_record, SMOKE_TEST)
prepared = prepared_splits(split_graphs, gnn, device)
train_graphs, val_graphs, test_graphs = (prepared[name] for name in ("train", "val", "test"))
artifacts = []
dependencies = {}
run_mode = context["run_mode"]
g = test_graphs[0]
full_mask = torch.ones(g["edge_index"].shape[1], dtype=torch.bool, device=device)
print("=== DEVICE CHECK ===")
print("GNN:", next(gnn.parameters()).device)
for key in ("x", "edge_index", "edge_attr", "emb", "ctx"):
    print(key + ":", g[key].device)
print("Single-halo full-graph prediction:", float(predict_graph(gnn, g, full_mask)))


=== DEVICE CHECK ===
GNN: cuda:0
x: cuda:0
edge_index: cuda:0
edge_attr: cuda:0
emb: cuda:0
ctx: cuda:0
Single-halo full-graph prediction: 13.002946853637695


In [9]:
cfg["rls"]["sparsity_mode"] = "pair_pl"
cfg["rls"]["policy_normalize"] = True
cfg["rls"].update(POLICY_OVERRIDES)
if cfg["rls"]["sparsity_mode"] != "pair_pl":
    raise ValueError("Notebook B requires the repaired pair_pl objective.")
policy_epochs = cfg["rls"]["epochs"] if POLICY_EPOCHS is None else POLICY_EPOCHS
stageb_epochs = cfg["rls"].get("stageb_epochs", 10) if STAGEB_EPOCHS is None else STAGEB_EPOCHS
if any(not isinstance(value, int) or isinstance(value, bool) or value < 1 for value in (policy_epochs, stageb_epochs)):
    raise ValueError("Policy and Stage B epochs must be positive integers.")
if SMOKE_TEST:
    policy_epochs, stageb_epochs = min(policy_epochs, 2), min(stageb_epochs, 2)
cfg["rls"].update(epochs=policy_epochs, stageb_epochs=stageb_epochs)
rls_cfg = cfg["rls"]
policy = build_policy(cfg, node_emb_dim=gnn.output_dim).to(device)
value_net = ValueNet(gnn.output_dim).to(device)
trainer = PolicyGradientTrainer(policy, value_net,
    torch.optim.Adam(policy.parameters(), lr=rls_cfg["lr"]),
    torch.optim.Adam(value_net.parameters(), lr=rls_cfg["lr"]), rls_cfg)
random_val_rows = random_pair_reference(val_graphs, gnn, rls_cfg,
    range(2 if SMOKE_TEST else int(rls_cfg.get("random_validation_seeds", 10))))
pd.DataFrame(random_val_rows).to_csv(OUT / "random_validation.csv", index=False)
log_rows = []
def training_log(epoch, target, loss):
    if not np.isfinite(loss):
        raise RuntimeError(f"Nonfinite training loss at epoch {epoch}; no completed policy will be published.")
    log_rows.append(dict(epoch=epoch, target_sp=target, loss=loss))
    print(f"Epoch {epoch + 1}/{policy_epochs}: target={target:.3f}, loss={loss:.4f}")
losses = train_policy(trainer, train_graphs, prediction_pair(gnn), rls_cfg,
                      device, epochs=policy_epochs, log_fn=training_log,
                      validation_fn=lambda p: policy_metrics(p, val_graphs, gnn, rls_cfg))
policy.eval()
selected_val = policy_metrics(policy, val_graphs, gnn, rls_cfg)
policy_quality = policy_validation(selected_val, random_val_rows)
policy_quality["selected_epoch"] = trainer.selected_epoch
policy_quality["scope"] = "smoke" if SMOKE_TEST else "validation"
policy_filename = "policy.pt" if policy_quality["verdict"] == "PASS" or SMOKE_TEST else "policy_diagnostic.pt"
# Delete only obsolete publishable artifacts from this output folder after a failed rerun.
if policy_filename != "policy.pt":
    (OUT / "policy.pt").unlink(missing_ok=True)
torch.save(policy.state_dict(), OUT / policy_filename)
print("Policy validation:", policy_quality)
torch.save(value_net.state_dict(), OUT / "value_net.pt")
pd.DataFrame(trainer.diagnostics).to_csv(OUT / "training_log.csv", index=False)
artifacts += [policy_filename, "value_net.pt", "training_log.csv", "random_validation.csv"]
(OUT / "policy_validation.json").write_text(json.dumps(policy_quality, indent=2))
artifacts.append("policy_validation.json")


Epoch 1/60: target=0.900, loss=0.1251
Epoch 2/60: target=0.888, loss=0.0122
Epoch 3/60: target=0.875, loss=0.0116
Epoch 4/60: target=0.863, loss=0.0028
Epoch 5/60: target=0.850, loss=-0.0143
Epoch 6/60: target=0.838, loss=0.0063
Epoch 7/60: target=0.825, loss=0.0036
Epoch 8/60: target=0.812, loss=-0.0185
Epoch 9/60: target=0.800, loss=-0.0135
Epoch 10/60: target=0.787, loss=-0.0176
Epoch 11/60: target=0.775, loss=-0.0236
Epoch 12/60: target=0.762, loss=0.0057
Epoch 13/60: target=0.750, loss=0.0100
Epoch 14/60: target=0.738, loss=0.0091
Epoch 15/60: target=0.725, loss=-0.0129
Epoch 16/60: target=0.713, loss=0.0031
Epoch 17/60: target=0.700, loss=0.0009
Epoch 18/60: target=0.688, loss=0.0062
Epoch 19/60: target=0.675, loss=0.0097
Epoch 20/60: target=0.663, loss=0.0123
Epoch 21/60: target=0.650, loss=-0.0016
Epoch 22/60: target=0.637, loss=-0.0007
Epoch 23/60: target=0.625, loss=-0.0099
Epoch 24/60: target=0.613, loss=0.0086
Epoch 25/60: target=0.600, loss=-0.0059
Epoch 26/60: target=0.58

In [10]:
stageb_model, stageb_record = None, None
stageb_info = {"enabled": RUN_STAGE_B}
if RUN_STAGE_B and (policy_quality["verdict"] == "PASS" or SMOKE_TEST):
    train_masks, _ = policy_masks(policy, train_graphs, rls_cfg)
    val_masks, _ = policy_masks(policy, val_graphs, rls_cfg)
    stageb_model = copy.deepcopy(gnn).requires_grad_(True)
    history, info = fine_tune_gnn(stageb_model, train_graphs, train_masks,
        epochs=stageb_epochs, lr=float(rls_cfg.get("stageb_lr", 1e-4)), device=device,
        val_graphs=val_graphs, val_masks=val_masks,
        patience=int(rls_cfg.get("stageb_patience", 3)),
        full_tol=float(rls_cfg.get("stageb_full_tol", 0.02)),
        full_loss_weight=float(rls_cfg.get("stageb_full_loss_weight", 0.5)))
    if not np.isfinite(history).all():
        raise RuntimeError("Stage B produced nonfinite losses.")
    # No admissible validation epoch: restore the original model.
    if info["best_epoch"] < 0:
        stageb_model.load_state_dict(gnn.state_dict())
    stageb_model.eval().requires_grad_(False)
    stageb_record = record_backbone("stageB_finetuned" if info["accepted"] else "frozen", stageb_model)
    stageb_info.update(info, restored_frozen=info["best_epoch"] < 0)
    if info["accepted"]:
        torch.save(stageb_model.state_dict(), OUT / "finetuned_gnn.pt")
        artifacts.append("finetuned_gnn.pt")
    else:
        (OUT / "finetuned_gnn.pt").unlink(missing_ok=True)
        print("No accepted Stage B model; original frozen backbone restored.")
    pd.DataFrame({"epoch": range(len(history)), "loss": history,
                  "val_full_rmse": info["full_hist"], "val_pruned_rmse": info["pruned_hist"]}).to_csv(OUT / "stageb_training_log.csv", index=False)
    artifacts += ["stageb_training_log.csv"]
    print("Stage B:", stageb_info)
else:
    (OUT / "finetuned_gnn.pt").unlink(missing_ok=True)
    print("Stage B skipped: disabled or policy validation failed.")


No accepted Stage B model; original frozen backbone restored.
Stage B: {'enabled': True, 'best_epoch': -1, 'stopped_early': True, 'full_hist': [0.1461620032787323, 0.11952973157167435, 0.1218491792678833], 'pruned_hist': [0.13665162026882172, 0.1268165409564972, 0.12189040333032608], 'best_full': 0.11749376356601715, 'best_pruned': 0.12099479883909225, 'prefinetune_full': 0.11749376356601715, 'prefinetune_pruned': 0.12099479883909225, 'accepted': False, 'restored_frozen': True}


In [11]:
test_masks, _ = policy_masks(policy, test_graphs, rls_cfg)
results = evaluate_masks(gnn, test_graphs, test_masks)
rows = result_rows(results, backbone_record, run_mode)
if stageb_model is not None and stageb_info.get("accepted", False):
    stageb_rows = result_rows(evaluate_masks(stageb_model, test_graphs, test_masks), stageb_record, run_mode)
    for row in stageb_rows:
        row["method"] += "_stageB"
    rows += stageb_rows
results_df = pd.DataFrame(rows)
results_df.to_csv(OUT / "policy_results.csv", index=False)
artifacts.append("policy_results.csv")
print(results_df.to_string(index=False))

validation_report = {"full": result_rows(evaluate_masks(gnn, val_graphs, policy_masks(policy, val_graphs, rls_cfg)[0]), backbone_record, run_mode)[0],
                     "policy": selected_val, "quality": policy_quality}
(OUT / "validation_report.json").write_text(json.dumps(validation_report, indent=2))
artifacts.append("validation_report.json")


   method     rmse       r2  scatter  mean_keep_frac  fidelity backbone_stage                                                  backbone_sha256        backbone run_mode  keep_frac
     full 0.116717 0.907471 0.117418        1.000000  1.000000         frozen 56907374794d9203b4b1873a3b930fff856aff427244d86e6a55742cded12034 frozen@56907374     full   1.000000
rl_policy 0.134322 0.877453 0.133531        0.479109  0.993169         frozen 56907374794d9203b4b1873a3b930fff856aff427244d86e6a55742cded12034 frozen@56907374     full   0.479109


In [12]:
if RUN_PENALTY_ABLATION:
    raise ValueError("Legacy Bernoulli ablation is not supported by the repaired Notebook B workflow.")
print("Legacy penalty ablation disabled.")


Legacy penalty ablation disabled.


In [13]:
assert record_backbone("frozen", gnn) == backbone_record, "Frozen backbone changed during this run."
context["config_used"] = copy.deepcopy(cfg)
manifest = complete_stage(OUT, "B", context, artifacts,
    repo={"head": REPO_HEAD, "branch": REPO_BRANCH}, graph_backend=builder.backend_used,
    full_split_sizes=full_split_sizes, device=str(device), torch=str(torch.__version__),
    dependencies=dependencies, stageb=stageb_info, stageb_backbone=stageb_record, policy_validation=policy_quality,
    policy_format="pair_pl_v1", reward_error_scale=rls_cfg["reward_error_scale"])


Notebook B completed: /kaggle/working/cosmic-net/outputs/rls/notebook_B
Download: /kaggle/working/cosmic-net/outputs/rls/notebook_B_full_outputs.zip
